# Experiment 9: Perceptron vs Multilayer Perceptron (A/B Experiment) with Hyperparameter Tuning

```
experiment9_pla_vs_mlp.py
=========================
ICS1512 - Machine Learning Algorithms Laboratory
Experiment 9 (Lab Manual title: "Perceptron vs Multilayer Perceptron
(A/B Experiment) with Hyperparameter Tuning")

Model A : Single-Layer Perceptron Learning Algorithm (PLA), implemented
          from scratch with a step activation and the classical
          w <- w + eta (y - y_hat) x update rule.
Model B : Multilayer Perceptron (scikit-learn MLPClassifier), tuned in
          four systematic stages (activation/optimizer, learning rate /
          batch size, architecture, regularisation).

Dataset : English Handwritten Characters (3,410 images, 62 classes).
Uses the reusable module ml_lab_utils.py (from Experiment 1) for:
    - the mandatory plot style    -> set_plot_style(), _bold_axis_labels()
    - 600 DPI EPS figure export   -> _save_eps()
```

In [ ]:
import json
import os
import time
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from PIL import Image

from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_curve, auc,
                             classification_report)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
IMG_SIZE = 32                      # images are resized to 32 x 32 -> 1024 features
FIG_DIR = "figures"
RES_DIR = "results"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

## 1. LOAD DATASET (extract the archive on first run)

In [ ]:
DATA_ZIP = "english_handwritten_characters.zip"
LABEL_CSV = "english.csv"

if not os.path.exists(LABEL_CSV) and os.path.exists(DATA_ZIP):
    with zipfile.ZipFile(DATA_ZIP) as zf:
        zf.extractall(".")
    print(f"Extracted {DATA_ZIP}")

labels_df = pd.read_csv(LABEL_CSV)
print(labels_df.shape)
print(labels_df.head())
print("Missing values:", int(labels_df.isna().sum().sum()))
print("Number of classes:", labels_df["label"].nunique())
print("Images per class: min = %d, max = %d" % (labels_df["label"].value_counts().min(),
                                                labels_df["label"].value_counts().max()))

## 2. PREPROCESSING (resize, flatten, normalize)

Each PNG is converted to 8-bit grayscale, resized to a fixed 32x32 grid,
scaled to [0, 1] and inverted so that ink = 1.0 and background = 0.0
(this matters for the perceptron: the informative pixels then carry the
large values). The 32x32 grid is finally flattened into a 1024-dimensional
feature vector, which is what both models consume.

In [ ]:
def load_images(df, img_size=IMG_SIZE):
    X = np.zeros((len(df), img_size * img_size), dtype=np.float32)
    raw_sizes = []
    for i, path in enumerate(df["image"].values):
        with Image.open(path) as im:
            raw_sizes.append(im.size)
            g = im.convert("L").resize((img_size, img_size), Image.BILINEAR)
        arr = np.asarray(g, dtype=np.float32) / 255.0
        X[i] = (1.0 - arr).ravel()          # invert: ink -> 1, paper -> 0
    return X, raw_sizes


t0 = time.time()
X, raw_sizes = load_images(labels_df)
classes = np.array(sorted(labels_df["label"].unique()))
class_to_idx = {c: i for i, c in enumerate(classes)}
y = labels_df["label"].map(class_to_idx).values
print(f"Loaded {X.shape[0]} images as {X.shape[1]}-dim vectors in {time.time() - t0:.1f}s")
print("Raw image sizes: %d unique, e.g. %s" % (len(set(raw_sizes)), sorted(set(raw_sizes))[:5]))
print("Pixel range after preprocessing: [%.2f, %.2f], mean ink = %.4f"
      % (X.min(), X.max(), X.mean()))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 3. EXPLORATORY DATA ANALYSIS

## Reusable utilities (`ml_lab_utils`, from Experiment 1)

Inlined here so this notebook runs on its own without a separate `ml_lab_utils.py`.

In [ ]:
"""
ml_lab_utils.py
================
ICS1512 - Machine Learning Algorithms Laboratory
Reusable utility module used across ALL experiments.

Implements (per lab manual, Section 4):
    1. One reusable EDA function            -> generate_eda_summary()
    2. One reusable Regression function      -> train_evaluate_regression()
    3. One reusable Classification function  -> train_evaluate_classification()
    4. One reusable Regression metrics fn    -> regression_performance_metrics()
    5. One reusable Classification metrics   -> classification_performance_metrics()

Formatting rules enforced everywhere (per lab manual, Section 1):
    - Times New Roman, 15 pt for all text / legends
    - Bold, Times New Roman, 15 pt axis labels
    - Figures exported as .eps at 600 DPI (Section 3)

NOTE on fonts: "Times New Roman" itself is a proprietary Microsoft font and is
not installable on Linux. Liberation Serif is metrically-compatible (identical
glyph widths/kerning) and is registered here under the family name
"Times New Roman" so that rcParams['font.family'] = 'Times New Roman' works
transparently. On Windows/macOS, if the real Times New Roman is installed,
matplotlib will simply use that instead.
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------
# 1. GLOBAL PLOT STYLE  (Section 1 of the manual)
# --------------------------------------------------------------------------
def set_plot_style(font_size=15):
    """
    Applies the mandatory lab formatting to every matplotlib figure:
        - Times New Roman (or metric-compatible Liberation Serif) font
        - 15 pt base font size
        - 15 pt Times New Roman legends
        - Bold, 15 pt, Times New Roman axis labels
    Call this once at the start of a notebook / script.
    """
    # Register Liberation Serif under the alias "Times New Roman" if the
    # genuine font is not present on this machine.
    installed_fonts = {f.name for f in fm.fontManager.ttflist}
    if "Times New Roman" not in installed_fonts:
        liberation_paths = [
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Bold.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Italic.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-BoldItalic.ttf",
        ]
        for p in liberation_paths:
            if os.path.exists(p):
                fm.fontManager.addfont(p)
                # Force the registered family name to "Times New Roman"
                # (FontEntry is a frozen dataclass in modern matplotlib, so we
                # replace the last-added entry rather than mutate it in place)
                last = fm.fontManager.ttflist[-1]
                fm.fontManager.ttflist[-1] = fm.FontEntry(
                    fname=last.fname, name="Times New Roman",
                    style=last.style, variant=last.variant,
                    weight=last.weight, stretch=last.stretch, size=last.size,
                )

    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": font_size,
        "legend.fontsize": font_size,
        "legend.title_fontsize": font_size,
        "axes.labelsize": font_size,
        "axes.labelweight": "bold",
        "axes.titlesize": font_size,
        "axes.titleweight": "bold",
        "xtick.labelsize": font_size - 2,
        "ytick.labelsize": font_size - 2,
        "figure.titlesize": font_size + 2,
        "savefig.dpi": 600,
        "figure.dpi": 150,   # screen preview; export always forced to 600 (see save)
        "svg.fonttype": "none",
    })


def _bold_axis_labels(ax, xlabel=None, ylabel=None, title=None, fs=15):
    """Helper: apply Times New Roman / Bold / 15pt to a single axis explicitly."""
    fp_bold = fm.FontProperties(family="Times New Roman", weight="bold", size=fs)
    fp_reg = fm.FontProperties(family="Times New Roman", size=fs - 2)
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontproperties=fp_bold)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontproperties=fp_bold)
    if title is not None:
        ax.set_title(title, fontproperties=fp_bold, fontsize=fs)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontproperties(fp_reg)
    leg = ax.get_legend()
    if leg is not None:
        for txt in leg.get_texts():
            txt.set_fontproperties(fp_reg)


def _save_eps(fig, save_path, also_png=True):
    """Export a figure as .eps at 600 DPI (Section 3 of the manual).

    If also_png is True, an additional .png copy is saved alongside the .eps
    (same basename) purely so the figure can be embedded when compiling the
    LaTeX report with pdflatex/xelatex, which cannot rasterize .eps directly
    without Ghostscript. The .eps remains the official, mandated deliverable.
    """
    if save_path is None:
        return None
    if not save_path.lower().endswith(".eps"):
        save_path = os.path.splitext(save_path)[0] + ".eps"
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig.savefig(save_path, format="eps", dpi=600, bbox_inches="tight")
    if also_png:
        png_path = os.path.splitext(save_path)[0] + ".png"
        fig.savefig(png_path, format="png", dpi=200, bbox_inches="tight")
    return save_path


# --------------------------------------------------------------------------
# 2. GENERIC EDA FUNCTION  (Section 4.1)  -> ONE consolidated 12-subplot figure
# --------------------------------------------------------------------------
def generate_eda_summary(df, target_col=None, dataset_name="Dataset",
                          save_path=None, figsize=(22, 16)):
    """
    Generic, reusable EDA function that works on ANY tabular dataset
    (classification, regression, or unlabeled). Produces ONE consolidated
    figure containing 12 EDA subplots on a single page, per Section 2 of the
    lab manual.

    Parameters
    ----------
    df : pandas.DataFrame
        The full dataset (features + target, if any).
    target_col : str or None
        Name of the target/label column, if present. If None, the function
        treats the dataset as unlabeled and adapts the 12-panel layout
        accordingly (no class-distribution / target-correlation panels).
    dataset_name : str
        Used in the figure's suptitle.
    save_path : str or None
        If given, the figure is exported as .eps @ 600 DPI to this path.
    figsize : tuple
        Overall figure size in inches.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    set_plot_style()
    df = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
    if target_col in numeric_cols:
        numeric_cols.remove(target_col)
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)

    is_classification_target = (
        target_col is not None and
        (df[target_col].dtype == "object" or df[target_col].nunique() <= 20)
    )

    # Pick the most "informative" numeric feature (highest variance) as the
    # representative single feature for panels 6/8/9/10, instead of blindly
    # using the first column (which can be degenerate/constant, e.g. corner
    # pixels in an image dataset such as MNIST/Digits).
    if numeric_cols:
        # Prefer genuinely continuous columns (more than 5 distinct values) so
        # binary/near-constant encoded columns (e.g. a 0/1 "sex" flag, or
        # constant corner pixels in image data) are not picked as the
        # representative single feature for panels 6/8/9/10.
        continuous_cols = [c for c in numeric_cols if df[c].nunique() > 5]
        candidate_cols = continuous_cols if continuous_cols else numeric_cols
        variances = df[candidate_cols].var().sort_values(ascending=False)
        top_var_cols = variances.index.tolist()
        feat_a = top_var_cols[0]
        feat_b = top_var_cols[1] if len(top_var_cols) > 1 else top_var_cols[0]
        kde_cols = top_var_cols[:4]
    else:
        feat_a = feat_b = None
        kde_cols = []

    fig = plt.figure(figsize=figsize)
    fig.suptitle(f"Exploratory Data Analysis Summary \u2013 {dataset_name}",
                 fontweight="bold", fontsize=17,
                 fontproperties=fm.FontProperties(family="Times New Roman",
                                                   weight="bold", size=17))
    gs = fig.add_gridspec(3, 4, hspace=0.55, wspace=0.4)
    axes = [fig.add_subplot(gs[i // 4, i % 4]) for i in range(12)]
    panel = 0

    # ---- Panel 1: Dataset overview (head / shape as a text table) ----
    ax = axes[panel]; panel += 1
    ax.axis("off")
    overview_txt = (
        f"Shape: {df.shape[0]} rows x {df.shape[1]} cols\n"
        f"Numeric features: {len(numeric_cols)}\n"
        f"Categorical features: {len(categorical_cols)}\n"
        f"Missing cells: {int(df.isnull().sum().sum())}\n"
        f"Duplicate rows: {int(df.duplicated().sum())}"
    )
    ax.text(0.02, 0.9, overview_txt, va="top", ha="left",
            fontproperties=fm.FontProperties(family="Times New Roman", size=13),
            transform=ax.transAxes)
    _bold_axis_labels(ax, title="1. Dataset Overview")

    # ---- Panel 2: Statistical summary heat-table (mean/std/min/max) ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().T[["mean", "std", "min", "max"]]
        desc_norm = (desc - desc.min()) / (desc.max() - desc.min() + 1e-9)
        sns.heatmap(desc_norm.iloc[:8], annot=desc.iloc[:8].round(1), fmt="",
                    cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontsize": 8, "fontfamily": "Times New Roman"})
    _bold_axis_labels(ax, title="2. Statistical Summary")

    # ---- Panel 3: Missing value analysis ----
    ax = axes[panel]; panel += 1
    miss = df.isnull().mean().sort_values(ascending=False) * 100
    if miss.sum() == 0:
        ax.text(0.5, 0.5, "No Missing Values", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=14))
        ax.axis("off")
    else:
        miss[miss > 0].head(10).plot(kind="bar", ax=ax, color="#c0392b")
    _bold_axis_labels(ax, "Feature", "% Missing", "3. Missing Value Analysis")

    # ---- Panel 4: Class distribution / target distribution ----
    ax = axes[panel]; panel += 1
    if target_col is not None:
        if is_classification_target:
            df[target_col].value_counts().plot(kind="bar", ax=ax, color="#2980b9")
            _bold_axis_labels(ax, "Class", "Count", "4. Class Distribution")
        else:
            sns.histplot(df[target_col], kde=True, ax=ax, color="#2980b9")
            _bold_axis_labels(ax, target_col, "Frequency", "4. Target Distribution")
    else:
        ax.axis("off")
        ax.text(0.5, 0.5, "No target column supplied", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=12))
        _bold_axis_labels(ax, title="4. Target Distribution")

    # ---- Panel 5: Correlation matrix (heatmap) ----
    ax = axes[panel]; panel += 1
    corr_cols = numeric_cols[:10] if len(numeric_cols) > 10 else numeric_cols
    if len(corr_cols) >= 2:
        sns.heatmap(df[corr_cols].corr(), cmap="coolwarm", center=0, ax=ax,
                    cbar=False, annot=len(corr_cols) <= 6, fmt=".2f",
                    annot_kws={"fontsize": 7})
    _bold_axis_labels(ax, title="5. Correlation Matrix")

    # ---- Panel 6: Feature distribution (histogram of 1st numeric feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.histplot(df[feat_a], kde=True, ax=ax, color="#27ae60")
    _bold_axis_labels(ax, feat_a if feat_a else "", "Frequency",
                       "6. Feature Distribution")

    # ---- Panel 7: Box plot (outlier detection) across numeric features ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        plot_cols = numeric_cols[:6]
        df_scaled = (df[plot_cols] - df[plot_cols].mean()) / (df[plot_cols].std() + 1e-9)
        sns.boxplot(data=df_scaled, ax=ax, color="#f39c12")
        ax.tick_params(axis="x", rotation=45)
    _bold_axis_labels(ax, "Feature", "Standardized Value", "7. Box Plot (Outliers)")

    # ---- Panel 8: Violin plot ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.violinplot(y=df[feat_a], ax=ax, color="#8e44ad")
    _bold_axis_labels(ax, "", feat_a if feat_a else "",
                       "8. Violin Plot")

    # ---- Panel 9: Scatter plot (feature 1 vs feature 2, hued by target) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None and feat_b is not None:
        hue = df[target_col] if (target_col and is_classification_target) else None
        sns.scatterplot(x=df[feat_a], y=df[feat_b],
                         hue=hue, ax=ax, palette="Set2", legend=False, s=18)
    _bold_axis_labels(ax, feat_a if feat_a else "", feat_b if feat_b else "",
                       "9. Scatter Plot")

    # ---- Panel 10: Q-Q plot (normality check on highest-variance feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        stats.probplot(df[feat_a].dropna(), dist="norm", plot=ax)
        ax.get_lines()[0].set_markerfacecolor("#2980b9")
        ax.get_lines()[0].set_markeredgecolor("#2980b9")
        ax.get_lines()[1].set_color("#c0392b")
    _bold_axis_labels(ax, "Theoretical Quantiles", "Sample Quantiles", "10. Q-Q Plot")

    # ---- Panel 11: KDE / density plot overlay of top numeric features ----
    ax = axes[panel]; panel += 1
    for c in kde_cols:
        sns.kdeplot(df[c], ax=ax, label=c, linewidth=1.5)
    if kde_cols:
        ax.legend(prop=fm.FontProperties(family="Times New Roman", size=9))
    _bold_axis_labels(ax, "Value", "Density", "11. KDE / Density Plot")

    # ---- Panel 12: Feature importance / variance plot ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        var = df[numeric_cols].var().sort_values(ascending=False).head(8)
        var.plot(kind="barh", ax=ax, color="#16a085")
        ax.invert_yaxis()
    _bold_axis_labels(ax, "Variance", "Feature", "12. Variance / Importance Plot")

    for ax in axes:
        _bold_axis_labels(ax)  # re-apply tick font in case a plotting call reset it

    saved = _save_eps(fig, save_path)
    if saved:
        print(f"[generate_eda_summary] Figure saved -> {saved} (600 DPI, EPS)")
    return fig


# --------------------------------------------------------------------------
# 3. GENERIC REGRESSION TRAIN/EVAL FUNCTION  (Section 4.2)
# --------------------------------------------------------------------------
def train_evaluate_regression(models: dict, X_train, X_test, y_train, y_test,
                               scale=False, verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of regression
    models on the same train/test split.

    Parameters
    ----------
    models : dict {name: sklearn-estimator}
    X_train, X_test, y_train, y_test : array-like
    scale : bool -> StandardScaler applied when True (fit on train only)
    verbose : bool -> print per-model metrics as they are computed

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by R2 desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = regression_performance_metrics(y_test, y_pred, model_name=name,
                                                   verbose=verbose, return_dict=True)
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("R2", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 4. GENERIC CLASSIFICATION TRAIN/EVAL FUNCTION  (Section 4.3)
# --------------------------------------------------------------------------
def train_evaluate_classification(models: dict, X_train, X_test, y_train, y_test,
                                   scale=False, average="weighted", verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of classification
    models on the same train/test split.

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by Accuracy desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = None
        if hasattr(model, "predict_proba"):
            try:
                y_proba = model.predict_proba(X_test)
            except Exception:
                y_proba = None
        metrics = classification_performance_metrics(
            y_test, y_pred, y_proba=y_proba, model_name=name,
            average=average, verbose=verbose, return_dict=True, plot=False
        )
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("Accuracy", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 5. GENERIC REGRESSION METRICS FUNCTION  (Section 4.4)
# --------------------------------------------------------------------------
def regression_performance_metrics(y_true, y_pred, model_name="Model",
                                    verbose=True, return_dict=False):
    """
    Computes and displays ALL standard regression performance metrics:
    MAE, MSE, RMSE, R2, Adjusted R2 (n only), MAPE.
    """
    from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                                  r2_score, mean_absolute_percentage_error)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    if verbose:
        print(f"--- Regression Metrics: {model_name} ---")
        print(f"  MAE  : {mae:.4f}")
        print(f"  MSE  : {mse:.4f}")
        print(f"  RMSE : {rmse:.4f}")
        print(f"  R2   : {r2:.4f}")
        print(f"  MAPE : {mape:.2f}%\n")

    result = {"Model": model_name, "MAE": mae, "MSE": mse,
              "RMSE": rmse, "R2": r2, "MAPE(%)": mape}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


# --------------------------------------------------------------------------
# 6. GENERIC CLASSIFICATION METRICS FUNCTION  (Section 4.5)
# --------------------------------------------------------------------------
def classification_performance_metrics(y_true, y_pred, y_proba=None,
                                        model_name="Model", average="weighted",
                                        verbose=True, return_dict=False,
                                        plot=True, save_path=None):
    """
    Computes and displays ALL standard classification performance metrics:
    Accuracy, Precision, Recall, F1-score, ROC-AUC (binary/multiclass ovr),
    and (optionally) plots the confusion matrix using the mandatory lab
    formatting (Times New Roman, bold 15pt axis labels).
    """
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                  f1_score, roc_auc_score, confusion_matrix)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=average, zero_division=0)
    rec = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=average, zero_division=0)

    roc_auc = np.nan
    if y_proba is not None:
        try:
            n_classes = y_proba.shape[1]
            if n_classes == 2:
                roc_auc = roc_auc_score(y_true, y_proba[:, 1])
            else:
                roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr",
                                         average=average)
        except Exception:
            roc_auc = np.nan

    if verbose:
        print(f"--- Classification Metrics: {model_name} ---")
        print(f"  Accuracy  : {acc:.4f}")
        print(f"  Precision : {prec:.4f}")
        print(f"  Recall    : {rec:.4f}")
        print(f"  F1-score  : {f1:.4f}")
        print(f"  ROC-AUC   : {roc_auc:.4f}" if not np.isnan(roc_auc) else "  ROC-AUC   : N/A")
        print()

    if plot:
        set_plot_style()
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontfamily": "Times New Roman", "fontsize": 13})
        _bold_axis_labels(ax, "Predicted Label", "True Label",
                           f"Confusion Matrix \u2013 {model_name}")
        _save_eps(fig, save_path)

    result = {"Model": model_name, "Accuracy": acc, "Precision": prec,
              "Recall": rec, "F1-score": f1, "ROC-AUC": roc_auc}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


In [ ]:
# ml_lab_utils (Experiment 1) provides set_plot_style / _bold_axis_labels / _save_eps.

set_plot_style()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Exploratory Data Analysis - English Handwritten Characters",
             fontfamily="Times New Roman", fontweight="bold")

# (1) sample characters
ax = axes[0, 0]
grid = np.zeros((IMG_SIZE * 4, IMG_SIZE * 8))
rng = np.random.RandomState(RANDOM_STATE)
picks = rng.choice(len(X), 32, replace=False)
for k, p in enumerate(picks):
    r, c = divmod(k, 8)
    grid[r * IMG_SIZE:(r + 1) * IMG_SIZE, c * IMG_SIZE:(c + 1) * IMG_SIZE] = X[p].reshape(IMG_SIZE, IMG_SIZE)
ax.imshow(grid, cmap="gray_r")
ax.set_xticks([]); ax.set_yticks([])
_bold_axis_labels(ax, None, None, "1. Sample characters (32x32, inverted)")

# (2) class distribution
ax = axes[0, 1]
counts = labels_df["label"].value_counts().sort_index()
ax.bar(range(len(counts)), counts.values, color="#4C72B0")
ax.set_xticks([0, 9, 35, 61])
ax.set_xticklabels(["0", "9", "Z", "z"])
_bold_axis_labels(ax, "Class (0-9, A-Z, a-z)", "Number of images",
                  "2. Class distribution (%d classes)" % len(counts))

# (3) mean character image
ax = axes[0, 2]
im = ax.imshow(X.mean(axis=0).reshape(IMG_SIZE, IMG_SIZE), cmap="viridis")
ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=ax, fraction=0.046)
_bold_axis_labels(ax, None, None, "3. Mean image (ink density)")

# (4) ink coverage per image
ax = axes[1, 0]
ink = X.mean(axis=1)
ax.hist(ink, bins=40, color="#55A868", edgecolor="black")
_bold_axis_labels(ax, "Mean ink fraction per image", "Frequency",
                  "4. Ink coverage distribution")

# (5) pixel variance map
ax = axes[1, 1]
im = ax.imshow(X.var(axis=0).reshape(IMG_SIZE, IMG_SIZE), cmap="magma")
ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=ax, fraction=0.046)
_bold_axis_labels(ax, None, None, "5. Per-pixel variance")

# (6) PCA projection coloured by character group
ax = axes[1, 2]
pca_vis = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
group = np.where(y < 10, 0, np.where(y < 36, 1, 2))
for g, name, col in [(0, "digits 0-9", "#C44E52"), (1, "upper A-Z", "#4C72B0"),
                     (2, "lower a-z", "#55A868")]:
    m = group == g
    ax.scatter(pca_vis[m, 0], pca_vis[m, 1], s=6, alpha=0.5, c=col, label=name)
ax.legend(markerscale=2)
_bold_axis_labels(ax, "PC 1", "PC 2", "6. PCA projection by character group")

fig.tight_layout()
_save_eps(fig, f"{FIG_DIR}/eda_handwritten_characters.eps")
plt.close(fig)

pca_full = PCA(n_components=0.95, random_state=RANDOM_STATE).fit(X_train)
print("Components for 95%% variance: %d of %d" % (pca_full.n_components_, X.shape[1]))

## 4. MODEL A - PERCEPTRON LEARNING ALGORITHM (from scratch)

One binary perceptron per class (one-vs-rest), step activation, and the
classical update rule $w \leftarrow w + \eta (y - \hat{y}) x$. Weights are
initialised from a small random normal so that the learning rate genuinely
changes the trajectory (with all-zero initialisation an OvR argmax
perceptron is invariant to eta).

In [ ]:
class PerceptronLearningAlgorithm:
    """Single-layer perceptron (one-vs-rest), step activation, from scratch."""

    def __init__(self, n_classes, eta=0.01, n_epochs=40, random_state=RANDOM_STATE):
        self.n_classes = n_classes
        self.eta = eta
        self.n_epochs = n_epochs
        self.random_state = random_state

    def _scores(self, X):
        return X @ self.W + self.b

    def fit(self, X, y, X_val=None, y_val=None):
        rng = np.random.RandomState(self.random_state)
        n_features = X.shape[1]
        self.W = rng.normal(0.0, 0.01, size=(n_features, self.n_classes))
        self.b = np.zeros(self.n_classes)
        Y = -np.ones((len(y), self.n_classes))         # one-vs-rest targets in {-1, +1}
        Y[np.arange(len(y)), y] = 1.0

        self.train_error_, self.val_error_ = [], []
        for _ in range(self.n_epochs):
            order = rng.permutation(len(X))
            for i in order:                            # online (sample-by-sample) updates
                xi = X[i]
                pred = np.where(xi @ self.W + self.b >= 0.0, 1.0, -1.0)  # step activation
                err = Y[i] - pred                      # 0 where correct, +-2 otherwise
                if np.any(err):
                    self.W += self.eta * np.outer(xi, err)
                    self.b += self.eta * err
            self.train_error_.append(1.0 - accuracy_score(y, self.predict(X)))
            if X_val is not None:
                self.val_error_.append(1.0 - accuracy_score(y_val, self.predict(X_val)))
        return self

    def predict(self, X):
        return np.argmax(self._scores(X), axis=1)

    def decision_function(self, X):
        return self._scores(X)


# Learning-rate sweep for the PLA (Model A's only real hyperparameter).
pla_runs, pla_models = [], {}
for eta in [0.001, 0.01, 0.1, 1.0]:
    t0 = time.time()
    pla = PerceptronLearningAlgorithm(len(classes), eta=eta, n_epochs=40)
    pla.fit(X_train, y_train, X_test, y_test)
    acc = accuracy_score(y_test, pla.predict(X_test))
    pla_runs.append({"eta": eta, "Train Error (final)": pla.train_error_[-1],
                     "Test Accuracy": acc, "Time (s)": time.time() - t0})
    pla_models[eta] = pla
    print(f"  PLA eta={eta}: test accuracy = {acc:.4f}  ({time.time() - t0:.1f}s)")

pla_runs_df = pd.DataFrame(pla_runs)
pla_runs_df.to_csv(f"{RES_DIR}/pla_learning_rate_sweep.csv", index=False)
best_eta = float(pla_runs_df.loc[pla_runs_df["Test Accuracy"].idxmax(), "eta"])
pla_best = pla_models[best_eta]
print(pla_runs_df)
print("Best PLA learning rate:", best_eta)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for eta, m in pla_models.items():
    ax.plot(range(1, len(m.train_error_) + 1), m.train_error_, label=f"$\\eta$ = {eta}")
ax.set_ylim(0, 1)
ax.legend(title="Learning rate")
_bold_axis_labels(ax, "Epoch", "Training error rate",
                  "PLA convergence: training error vs epochs")
_save_eps(fig, f"{FIG_DIR}/pla_convergence.eps")
plt.close(fig)

## 5. MODEL B - MLP HYPERPARAMETER TUNING (four systematic stages)

In [ ]:
MLP_BASE = dict(hidden_layer_sizes=(256,), activation="relu", solver="adam",
                learning_rate_init=1e-3, batch_size=64, max_iter=60,
                random_state=RANDOM_STATE, early_stopping=False)


def tune(stage_name, grid, base):
    t0 = time.time()
    gs = GridSearchCV(MLPClassifier(**base), grid, cv=3, scoring="accuracy",
                      n_jobs=-1, return_train_score=False)
    gs.fit(X_train, y_train)
    res = pd.DataFrame(gs.cv_results_)[["params", "mean_test_score", "std_test_score"]]
    res = res.sort_values("mean_test_score", ascending=False).reset_index(drop=True)
    res.insert(0, "stage", stage_name)
    print(f"\n--- {stage_name} (best CV accuracy {gs.best_score_:.4f}, "
          f"{time.time() - t0:.0f}s) ---")
    print(res.head(10).to_string(index=False))
    return gs, res


base = dict(MLP_BASE)
gs1, res1 = tune("1. Activation x Optimizer",
                 {"activation": ["relu", "tanh", "logistic"],
                  "solver": ["adam", "sgd"]}, base)
base.update(gs1.best_params_)

gs2, res2 = tune("2. Learning rate x Batch size",
                 {"learning_rate_init": [1e-4, 1e-3, 1e-2],
                  "batch_size": [32, 64, 128]}, base)
base.update(gs2.best_params_)

gs3, res3 = tune("3. Architecture",
                 {"hidden_layer_sizes": [(128,), (256,), (512,), (256, 128), (512, 256)]},
                 base)
base.update(gs3.best_params_)

# Stage 4 exists because stages 1-3 drove training accuracy to ~100% while test
# accuracy stalled: L2 penalty and early stopping are the two knobs that act
# directly on that gap.
gs4, res4 = tune("4. Regularisation",
                 {"alpha": [1e-4, 1e-2, 1e-1, 1.0],
                  "early_stopping": [False, True]}, base)
base.update(gs4.best_params_)

tuning_df = pd.concat([res1, res2, res3, res4], ignore_index=True)
tuning_df["params"] = tuning_df["params"].astype(str)
tuning_df.to_csv(f"{RES_DIR}/mlp_hyperparameter_tuning.csv", index=False)
print("\nFinal chosen MLP configuration:")
final_cfg = {k: base[k] for k in ["hidden_layer_sizes", "activation", "solver",
                                  "learning_rate_init", "batch_size", "alpha",
                                  "early_stopping"]}
print(final_cfg)

## 6. TRAIN THE FINAL MODELS

In [ ]:
t0 = time.time()
mlp = MLPClassifier(**{**base, "max_iter": 300, "n_iter_no_change": 15})
mlp.fit(X_train, y_train)
mlp_train_time = time.time() - t0
print(f"Final MLP: {mlp.n_iter_} iterations, {mlp_train_time:.1f}s, "
      f"final training loss = {mlp.loss_:.4f}")

# an untuned baseline MLP, to quantify what the tuning actually bought
mlp_base_model = MLPClassifier(**{**MLP_BASE, "max_iter": 300, "n_iter_no_change": 15})
mlp_base_model.fit(X_train, y_train)
print("Baseline (untuned) MLP test accuracy:",
      accuracy_score(y_test, mlp_base_model.predict(X_test)))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(range(1, len(mlp.loss_curve_) + 1), mlp.loss_curve_,
        color="#4C72B0", label="Tuned MLP")
ax.plot(range(1, len(mlp_base_model.loss_curve_) + 1), mlp_base_model.loss_curve_,
        color="#C44E52", linestyle="--", label="Untuned baseline MLP")
ax.legend()
_bold_axis_labels(ax, "Epoch", "Cross-entropy loss",
                  "MLP convergence: training loss vs epochs")
_save_eps(fig, f"{FIG_DIR}/mlp_loss_curve.eps")
plt.close(fig)

## 7. A/B COMPARISON - EVALUATION METRICS

In [ ]:
def evaluate(name, y_true, y_pred, scores, train_time):
    return {"Model": name,
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "F1-score": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "Training Time (s)": train_time}


t0 = time.time()
PerceptronLearningAlgorithm(len(classes), eta=best_eta, n_epochs=40).fit(X_train, y_train)
pla_train_time = time.time() - t0

y_pred_pla = pla_best.predict(X_test)
y_pred_mlp = mlp.predict(X_test)
scores_pla = pla_best.decision_function(X_test)
proba_mlp = mlp.predict_proba(X_test)

comparison = pd.DataFrame([
    evaluate("PLA (Model A)", y_test, y_pred_pla, scores_pla, pla_train_time),
    evaluate("MLP (Model B, tuned)", y_test, y_pred_mlp, proba_mlp, mlp_train_time),
])
comparison.to_csv(f"{RES_DIR}/ab_comparison.csv", index=False)
print(comparison.to_string(index=False))

In [ ]:
# ROC curves (one-vs-rest, micro and macro averaged over the 62 classes)
y_test_bin = label_binarize(y_test, classes=np.arange(len(classes)))


def roc_micro_macro(y_bin, scores):
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(y_bin.shape[1]):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], scores[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), scores.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(y_bin.shape[1])]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(y_bin.shape[1]):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= y_bin.shape[1]
    fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
    roc_auc["macro"] = auc(all_fpr, mean_tpr)
    return fpr, tpr, roc_auc


fpr_p, tpr_p, auc_p = roc_micro_macro(y_test_bin, scores_pla)
fpr_m, tpr_m, auc_m = roc_micro_macro(y_test_bin, proba_mlp)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (fpr, tpr, roc_auc, title) in zip(
        axes, [(fpr_p, tpr_p, auc_p, "PLA (Model A)"),
               (fpr_m, tpr_m, auc_m, "MLP (Model B, tuned)")]):
    ax.plot(fpr["micro"], tpr["micro"], color="#4C72B0",
            label=f"micro-average (AUC = {roc_auc['micro']:.3f})")
    ax.plot(fpr["macro"], tpr["macro"], color="#C44E52", linestyle="--",
            label=f"macro-average (AUC = {roc_auc['macro']:.3f})")
    ax.plot([0, 1], [0, 1], "k:", linewidth=1)
    ax.legend(loc="lower right")
    _bold_axis_labels(ax, "False Positive Rate", "True Positive Rate",
                      f"ROC - {title}")
fig.tight_layout()
_save_eps(fig, f"{FIG_DIR}/roc_curves.eps")
plt.close(fig)
print("ROC-AUC  PLA: micro %.4f / macro %.4f" % (auc_p["micro"], auc_p["macro"]))
print("ROC-AUC  MLP: micro %.4f / macro %.4f" % (auc_m["micro"], auc_m["macro"]))

In [ ]:
# Confusion matrices (62 x 62)
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
for ax, (y_pred, title) in zip(axes, [(y_pred_pla, "PLA (Model A)"),
                                      (y_pred_mlp, "MLP (Model B, tuned)")]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, cmap="Blues", cbar=True, ax=ax, square=True,
                xticklabels=classes, yticklabels=classes)
    ax.tick_params(labelsize=7)
    _bold_axis_labels(ax, "Predicted Label", "True Label",
                      f"Confusion Matrix - {title}")
fig.tight_layout()
_save_eps(fig, f"{FIG_DIR}/confusion_matrices.eps")
plt.close(fig)

In [ ]:
# Metric comparison bar chart + per-class F1
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
xs = np.arange(len(metrics)); w = 0.35
ax = axes[0]
ax.bar(xs - w / 2, comparison.loc[0, metrics].values.astype(float), w,
       label="PLA (Model A)", color="#C44E52")
ax.bar(xs + w / 2, comparison.loc[1, metrics].values.astype(float), w,
       label="MLP (Model B)", color="#4C72B0")
ax.set_xticks(xs); ax.set_xticklabels(metrics)
ax.legend()
_bold_axis_labels(ax, "Metric", "Score", "A/B comparison: PLA vs tuned MLP")

ax = axes[1]
f1_pla = f1_score(y_test, y_pred_pla, average=None, zero_division=0)
f1_mlp = f1_score(y_test, y_pred_mlp, average=None, zero_division=0)
ax.plot(range(len(classes)), f1_pla, color="#C44E52", label="PLA", linewidth=1)
ax.plot(range(len(classes)), f1_mlp, color="#4C72B0", label="MLP", linewidth=1)
ax.set_xticks([0, 9, 35, 61]); ax.set_xticklabels(["0", "9", "Z", "z"])
ax.legend()
_bold_axis_labels(ax, "Class (0-9, A-Z, a-z)", "Per-class F1-score",
                  "Per-class F1: PLA vs tuned MLP")
fig.tight_layout()
_save_eps(fig, f"{FIG_DIR}/model_comparison.eps")
plt.close(fig)

In [ ]:
# Tuning-stage figures
fig, axes = plt.subplots(1, 4, figsize=(26, 5.5))
for ax, (res, title, key) in zip(axes, [
        (res1, "Stage 1: activation x optimizer", None),
        (res2, "Stage 2: learning rate x batch size", None),
        (res3, "Stage 3: architecture", None),
        (res4, "Stage 4: regularisation", None)]):
    labels = [str(p).replace("{", "").replace("}", "").replace("'", "")
                .replace("hidden_layer_sizes: ", "").replace("learning_rate_init: ", "lr=")
                .replace("batch_size: ", "bs=").replace("activation: ", "")
                .replace("solver: ", "").replace("early_stopping: ", "early_stop=")
                .replace("alpha: ", "alpha=") for p in res["params"]]
    ax.barh(range(len(res)), res["mean_test_score"], color="#4C72B0")
    ax.set_yticks(range(len(res))); ax.set_yticklabels(labels, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlim(0, max(0.9, res["mean_test_score"].max() * 1.15))
    _bold_axis_labels(ax, "3-fold CV accuracy", None, title)
fig.tight_layout()
_save_eps(fig, f"{FIG_DIR}/mlp_tuning_stages.eps")
plt.close(fig)

## 8. SUMMARY OF NUMBERS USED IN THE REPORT

In [ ]:
summary = {
    "n_samples": int(len(X)), "n_features": int(X.shape[1]),
    "n_classes": int(len(classes)),
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "pca_components_95": int(pca_full.n_components_),
    "pla_best_eta": best_eta,
    "pla_learning_rate_sweep": pla_runs_df.to_dict("records"),
    "pla_final_train_error": float(pla_best.train_error_[-1]),
    "pla_epochs": int(len(pla_best.train_error_)),
    "mlp_final_config": {k: str(v) for k, v in final_cfg.items()},
    "mlp_stage_best": {"stage1": {str(k): str(v) for k, v in gs1.best_params_.items()},
                       "stage1_score": float(gs1.best_score_),
                       "stage2": {str(k): str(v) for k, v in gs2.best_params_.items()},
                       "stage2_score": float(gs2.best_score_),
                       "stage3": {str(k): str(v) for k, v in gs3.best_params_.items()},
                       "stage3_score": float(gs3.best_score_),
                       "stage4": {str(k): str(v) for k, v in gs4.best_params_.items()},
                       "stage4_score": float(gs4.best_score_)},
    "mlp_iterations": int(mlp.n_iter_), "mlp_final_loss": float(mlp.loss_),
    "mlp_baseline_test_accuracy": float(accuracy_score(y_test, mlp_base_model.predict(X_test))),
    "mlp_train_accuracy": float(accuracy_score(y_train, mlp.predict(X_train))),
    "pla_train_accuracy": float(accuracy_score(y_train, pla_best.predict(X_train))),
    "comparison": comparison.to_dict("records"),
    "roc": {"pla_micro": float(auc_p["micro"]), "pla_macro": float(auc_p["macro"]),
            "mlp_micro": float(auc_m["micro"]), "mlp_macro": float(auc_m["macro"])},
    "worst_classes_mlp": [str(classes[i]) for i in np.argsort(f1_mlp)[:6]],
    "best_classes_mlp": [str(classes[i]) for i in np.argsort(f1_mlp)[-6:]],
    "digits_vs_letters_f1_mlp": {
        "digits": float(np.mean(f1_mlp[:10])), "upper": float(np.mean(f1_mlp[10:36])),
        "lower": float(np.mean(f1_mlp[36:]))},
}
with open(f"{RES_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2)[:4000])

In [ ]:
print(classification_report(y_test, y_pred_mlp, target_names=[str(c) for c in classes],
                            zero_division=0))